# Starlink visibility tracker

Refactored workflow:
1. Load Starlink TLEs when a fresh sample is needed.
2. Generate visibility samples in local ENU coordinates.
3. Track a rolling top-k satellite set by slant-range rank.
4. Export a static sky plot and an animated GIF with colored top-k satellites and a legend.


In [1]:
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
import os

# Keep matplotlib/font caches in writable locations for notebook and batch runs.
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.animation import FuncAnimation, PillowWriter

# Make relative paths work whether the notebook kernel starts in this folder
# or in the parent project folder.
BASE_DIR = Path.cwd()
if not (BASE_DIR / "starlink_tracker.ipynb").exists():
    candidate = Path("/home/sj4025/my_project/Anti-jamming-unsync")
    if candidate.exists():
        BASE_DIR = candidate

RESULT_DIR = BASE_DIR / "result_plot"


@dataclass(frozen=True)
class TrackerConfig:
    lat_deg: float = 40.0822
    lon_deg: float = -105.1092
    elev_m: float = 1560.0

    elev_min_deg: float = 45.0
    az_range_deg: tuple = (0.0, 360.0)
    max_slant_km: float = 9999.0

    mode: str = "timeseries"      # "timeseries" or "random"
    duration_sec: int = 5 * 60      # used only when mode="timeseries"
    interval_sec: int = 10          # used only when mode="timeseries"
    range_hours: float = 24.0       # used only when mode="random"
    n_samples: int = 400            # used only when mode="random"
    rng_seed: int | None = None
    start_time_utc: datetime | None = None

    top_k: int = 5
    track_tol: int = 0

    tle_url: str = "https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle"
    user_agent: str = "sat-tracker/1.0"

    visibility_csv: Path = BASE_DIR / "starlink_timeseries_5min_all.csv"
    tracked_csv: Path = BASE_DIR / "starlink_timeseries_5min_top5.csv"
    static_png: Path = RESULT_DIR / "starlink_sky_topk.png"
    static_pdf: Path = RESULT_DIR / "starlink_sky_topk.pdf"
    animation_gif: Path = RESULT_DIR / "starlink_sky_topk.gif"


cfg = TrackerConfig()
cfg


TrackerConfig(lat_deg=40.0822, lon_deg=-105.1092, elev_m=1560.0, elev_min_deg=45.0, az_range_deg=(0.0, 360.0), max_slant_km=9999.0, mode='timeseries', duration_sec=300, interval_sec=10, range_hours=24.0, n_samples=400, rng_seed=None, start_time_utc=None, top_k=5, track_tol=0, tle_url='https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle', user_agent='sat-tracker/1.0', visibility_csv=PosixPath('/home/sj4025/my_project/Anti-jamming-unsync/starlink_timeseries_5min_all.csv'), tracked_csv=PosixPath('/home/sj4025/my_project/Anti-jamming-unsync/starlink_timeseries_5min_top5.csv'), static_png=PosixPath('/home/sj4025/my_project/Anti-jamming-unsync/result_plot/starlink_sky_topk.png'), static_pdf=PosixPath('/home/sj4025/my_project/Anti-jamming-unsync/result_plot/starlink_sky_topk.pdf'), animation_gif=PosixPath('/home/sj4025/my_project/Anti-jamming-unsync/result_plot/starlink_sky_topk.gif'))

In [2]:
def make_timescale():
    try:
        from skyfield.api import load
    except ImportError as exc:
        raise ImportError("skyfield is required only when RUN_SAMPLE=True. Install it to refresh TLE samples.") from exc
    return load.timescale()


def load_starlink_tles(timescale=None, url=cfg.tle_url, user_agent=cfg.user_agent, timeout=10):
    """Download Starlink TLEs from Celestrak and parse them as Skyfield satellites."""
    try:
        import requests
        from skyfield.api import EarthSatellite
    except ImportError as exc:
        raise ImportError("requests and skyfield are required only when RUN_SAMPLE=True.") from exc

    if timescale is None:
        timescale = make_timescale()

    response = requests.get(url, headers={"User-Agent": user_agent}, timeout=timeout)
    response.raise_for_status()
    tle_lines = response.text.strip().splitlines()

    satellites = []
    for i in range(0, len(tle_lines) - 2, 3):
        name = tle_lines[i].strip()
        line1 = tle_lines[i + 1].strip()
        line2 = tle_lines[i + 2].strip()
        if not (line1.startswith("1 ") and line2.startswith("2 ")):
            continue
        try:
            satellites.append((name, EarthSatellite(line1, line2, name, timescale)))
        except ValueError:
            continue

    if not satellites:
        raise RuntimeError("No valid Starlink TLE entries were parsed.")
    return satellites


def ensure_utc(dt=None):
    if dt is None:
        return datetime.now(tz=timezone.utc)
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def ecef_to_enu_rotation(lat_deg, lon_deg):
    phi = np.deg2rad(lat_deg)
    lam = np.deg2rad(lon_deg)
    return np.array([
        [-np.sin(lam),              np.cos(lam),               0.0],
        [-np.sin(phi) * np.cos(lam), -np.sin(phi) * np.sin(lam), np.cos(phi)],
        [ np.cos(phi) * np.cos(lam),  np.cos(phi) * np.sin(lam), np.sin(phi)],
    ])


def azimuth_in_range(az_deg, az_range_deg):
    az = az_deg % 360.0
    lo, hi = (float(v) % 360.0 for v in az_range_deg)
    if lo == hi:
        return True
    if lo < hi:
        return lo <= az < hi
    return az >= lo or az < hi


def make_time_samples(config):
    t0 = ensure_utc(config.start_time_utc)

    if config.mode == "timeseries":
        return [
            t0 + timedelta(seconds=s)
            for s in range(0, int(config.duration_sec), int(config.interval_sec))
        ]

    if config.mode == "random":
        rng = np.random.default_rng(config.rng_seed)
        offsets = np.sort(rng.uniform(0.0, float(config.range_hours) * 3600.0, size=int(config.n_samples)))
        return [t0 + timedelta(seconds=float(s)) for s in offsets]

    raise ValueError("config.mode must be 'timeseries' or 'random'.")


def sort_visibility(df):
    df = df.copy()
    time_key = pd.to_datetime(df["Time"], errors="coerce")
    if time_key.notna().all():
        df["_time_sort"] = time_key
        df = df.sort_values(["_time_sort", "Slant km", "Name"]).drop(columns="_time_sort")
    else:
        df = df.sort_values(["Time", "Slant km", "Name"])
    return df.reset_index(drop=True)


def collect_starlink_samples(satellites, timescale=None, config=cfg, out_csv=None):
    """Collect visible Starlink samples and return ENU coordinates relative to the ground station."""
    try:
        from skyfield.api import wgs84
        from skyfield.framelib import itrs
    except ImportError as exc:
        raise ImportError("skyfield is required only when RUN_SAMPLE=True. Install it to refresh TLE samples.") from exc

    if timescale is None:
        timescale = make_timescale()

    topos = wgs84.latlon(config.lat_deg, config.lon_deg, config.elev_m)
    rotation = ecef_to_enu_rotation(config.lat_deg, config.lon_deg)
    rows = []

    for dt_utc in make_time_samples(config):
        t = timescale.utc(dt_utc)
        obs_ecef = topos.at(t).frame_xyz(itrs).m
        if not np.isfinite(obs_ecef).all():
            continue

        time_label = dt_utc.strftime("%Y-%m-%d %H:%M:%S")
        for name, sat in satellites:
            try:
                alt, az, dist = (sat - topos).at(t).altaz()
                alt_deg = float(alt.degrees)
                az_deg = float(az.degrees)
                slant_km = float(dist.km)

                if not np.isfinite([alt_deg, az_deg, slant_km]).all():
                    continue
                if alt_deg < config.elev_min_deg:
                    continue
                if not azimuth_in_range(az_deg, config.az_range_deg):
                    continue
                if slant_km > config.max_slant_km:
                    continue

                sat_ecef = sat.at(t).frame_xyz(itrs).m
                if not np.isfinite(sat_ecef).all():
                    continue

                enu = rotation @ (sat_ecef - obs_ecef)
                if not np.isfinite(enu).all():
                    continue

                rows.append({
                    "Time": time_label,
                    "Name": name,
                    "Azimuth (°)": round(az_deg, 2),
                    "Elevation (°)": round(alt_deg, 2),
                    "Slant km": round(slant_km, 1),
                    "x_East (m)": round(float(enu[0]), 1),
                    "y_North (m)": round(float(enu[1]), 1),
                    "z_Up (m)": round(float(enu[2]), 1),
                })
            except Exception:
                continue

    columns = [
        "Time", "Name", "Azimuth (°)", "Elevation (°)", "Slant km",
        "x_East (m)", "y_North (m)", "z_Up (m)",
    ]
    df = pd.DataFrame(rows, columns=columns)
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    df = sort_visibility(df)

    if out_csv is not None:
        out_path = Path(out_csv)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_path, index=False)
    return df


In [3]:
def load_visibility_frame(data):
    if isinstance(data, pd.DataFrame):
        return sort_visibility(data)
    return sort_visibility(pd.read_csv(Path(data)))


def track_top_k(data, k=cfg.top_k, tol=cfg.track_tol, out_csv=None):
    """Track a rolling top-k set by slant-range rank, with optional rank tolerance."""
    if tol < 0:
        raise ValueError("tol must be >= 0")

    df = load_visibility_frame(data)
    df["Rank"] = df.groupby("Time")["Slant km"].rank(method="first", ascending=True).astype(int)

    selected_names = []
    out_rows = []

    for time_value, cur in df.groupby("Time", sort=False):
        cur = cur.sort_values(["Slant km", "Name"]).drop_duplicates("Name").reset_index(drop=True)
        by_name = cur.set_index("Name", drop=False)

        if not selected_names:
            selected = cur.head(k)["Name"].tolist()
            actions = ["seed"] * len(selected)
        else:
            kept = [
                name for name in selected_names
                if name in by_name.index and int(by_name.loc[name, "Rank"]) <= k + tol
            ]
            if not kept:
                selected = cur.head(k)["Name"].tolist()
                actions = ["reseed"] * len(selected)
            else:
                added = [name for name in cur["Name"].tolist() if name not in kept][:max(0, k - len(kept))]
                selected = kept + added
                actions = ["keep"] * len(kept) + ["add"] * len(added)

        selected_names = selected

        for name, action in zip(selected, actions):
            row = by_name.loc[name]
            out_rows.append({
                "Time": time_value,
                "Name": name,
                "Action": action,
                "Rank": int(row["Rank"]),
                "Slant km": row["Slant km"],
                "Azimuth (°)": row["Azimuth (°)"],
                "Elevation (°)": row["Elevation (°)"],
                "x_East (m)": row["x_East (m)"],
                "y_North (m)": row["y_North (m)"],
                "z_Up (m)": row["z_Up (m)"],
            })

    out = pd.DataFrame(out_rows)
    out = sort_visibility(out) if not out.empty else out

    if out_csv is not None:
        out_path = Path(out_csv)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out.to_csv(out_path, index=False)
    return out


In [ ]:
def display_sat_name(name):
    return str(name).replace(" [DTC]", "").replace("[DTC]", "")


def get_colors(names, cmap_name="tab10"):
    unique_names = list(dict.fromkeys(names))
    cmap = mpl.colormaps[cmap_name]
    return {name: cmap(i % cmap.N) for i, name in enumerate(unique_names)}


def polar_offsets(df, elev_min_deg):
    if df.empty:
        return np.empty((0, 2))
    elev = df["Elevation (°)"].clip(elev_min_deg, 90.0).to_numpy()
    theta = np.deg2rad(df["Azimuth (°)"].to_numpy())
    radius = 90.0 - elev
    return np.column_stack([theta, radius])


def style_sky_axis(ax, elev_min_deg, tick_step_deg=30, r_step_deg=10):
    r_outer = 90.0 - elev_min_deg
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_rlim(0, r_outer + 2)
    ax.set_axisbelow(True)
    ax.set_facecolor("white")
    ax.grid(True, linestyle="--", linewidth=0.7, alpha=0.5)

    rticks = np.arange(r_step_deg, r_outer + 0.1, r_step_deg)
    ax.set_yticks(rticks)
    ax.set_yticklabels([f"{90.0 - r:.0f}°" for r in rticks])
    ax.set_rlabel_position(45)
    az_ticks = np.arange(0, 360, tick_step_deg)
    ax.set_thetagrids(az_ticks, labels=[f"{a}°" for a in az_ticks])
    return ax


def select_snapshot(df, time=None, random_seed=None):
    grouped = df.groupby("Time", sort=False)
    time_points = list(grouped.groups.keys())
    if not time_points:
        raise ValueError("No time points are available for plotting.")

    if time is None and random_seed is None:
        time = time_points[0]
    elif time is None:
        rng = np.random.default_rng(random_seed)
        time = rng.choice(time_points)
    elif time not in grouped.groups:
        raise ValueError(f"Time {time!r} is not present in the visibility data.")

    return time, grouped.get_group(time).sort_values(["Slant km", "Name"]).reset_index(drop=True)


def save_static_sky_plot(
    visibility_data,
    out_png=cfg.static_png,
    out_pdf=cfg.static_pdf,
    k=cfg.top_k,
    elev_min_deg=cfg.elev_min_deg,
    time=None,
    random_seed=None,
    title=None,
    figsize=(3.35, 3.35),
    dpi=800,
):
    df = load_visibility_frame(visibility_data)
    time_value, data = select_snapshot(df, time=time, random_seed=random_seed)
    top = data.head(k).copy()
    others = data.iloc[k:].copy()
    colors = get_colors(top["Name"].tolist(), cmap_name="tab10")

    rc = {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.size": 8.0,
        "axes.labelsize": 8.0,
        "xtick.labelsize": 7.4,
        "ytick.labelsize": 7.4,
    }

    with mpl.rc_context(rc):
        fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=figsize)
        style_sky_axis(ax, elev_min_deg)
        ax.scatter(*polar_offsets(others, elev_min_deg).T, s=24, c="gray", alpha=0.65, linewidths=0, zorder=1)

        top_offsets = polar_offsets(top, elev_min_deg)
        for (theta, radius), name in zip(top_offsets, top["Name"]):
            ax.scatter([theta], [radius], s=62, c=[colors[name]], linewidths=0, zorder=3)

        legend_handles = [
            plt.Line2D(
                [0], [0], marker="o", linestyle="",
                markerfacecolor=colors[name], markeredgecolor="none",
                markersize=5, label=display_sat_name(name),
            )
            for name in top["Name"]
        ]
        ax.legend(
            handles=legend_handles,
            loc="upper left",
            bbox_to_anchor=(0.98, 1.10),
            borderaxespad=0.0,
            frameon=False,
            fontsize=7.0,
            handletextpad=0.25,
            labelspacing=0.25,
        )

        if title is not None:
            ax.set_title(title.format(time=time_value, k=k), pad=10)

        out_png = Path(out_png)
        out_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_png, dpi=dpi, bbox_inches="tight")
        if out_pdf is not None:
            out_pdf = Path(out_pdf)
            out_pdf.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(out_pdf, dpi=dpi, bbox_inches="tight")
        plt.close(fig)

    return {"time": time_value, "png": Path(out_png), "pdf": Path(out_pdf) if out_pdf else None}


def selected_top_for_time(data, tracked_group, time_value, k):
    if tracked_group is None or time_value not in tracked_group.groups:
        return data.head(k).copy()

    tracked = tracked_group.get_group(time_value).sort_values(["Rank", "Name"])
    visible_names = set(data["Name"])
    ordered_names = [name for name in tracked["Name"].tolist() if name in visible_names]
    by_name = data.set_index("Name", drop=False)
    return by_name.loc[ordered_names].reset_index(drop=True) if ordered_names else data.head(k).copy()


def save_sky_animation(
    visibility_data,
    out_gif=cfg.animation_gif,
    tracked_data=None,
    k=cfg.top_k,
    elev_min_deg=cfg.elev_min_deg,
    interval_ms=250,
    fps=4,
    dpi=140,
    figsize=(7.6, 6.0),
    title_template="Time: {time} | colored: top-{k}",
):
    df = load_visibility_frame(visibility_data)
    grouped = df.groupby("Time", sort=False)
    time_points = list(grouped.groups.keys())
    if not time_points:
        raise ValueError("No time points are available for animation.")

    tracked_group = None
    if tracked_data is not None:
        tracked_df = load_visibility_frame(tracked_data)
        tracked_group = tracked_df.groupby("Time", sort=False)
        color_names = tracked_df["Name"].tolist()
    else:
        color_names = []
        for _, cur in grouped:
            color_names.extend(cur.sort_values(["Slant km", "Name"]).head(k)["Name"].tolist())
    colors = get_colors(color_names, cmap_name="tab20")

    fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=figsize)
    fig.subplots_adjust(left=0.06, right=0.67, top=0.88, bottom=0.06)
    style_sky_axis(ax, elev_min_deg)
    scat_top = ax.scatter([], [], s=64, linewidths=0, zorder=3)
    scat_other = ax.scatter([], [], s=26, c="gray", alpha=0.55, linewidths=0, zorder=1)
    title_text = ax.set_title("", fontsize=10, pad=12)
    legend_ref = {"legend": None}

    def update(frame_idx):
        time_value = time_points[frame_idx]
        data = grouped.get_group(time_value).sort_values(["Slant km", "Name"]).reset_index(drop=True)
        top = selected_top_for_time(data, tracked_group, time_value, k)
        top_names = top["Name"].tolist()
        others = data[~data["Name"].isin(top_names)].copy()

        scat_other.set_offsets(polar_offsets(others, elev_min_deg))
        scat_top.set_offsets(polar_offsets(top, elev_min_deg))
        scat_top.set_facecolors([colors.get(name, "tab:blue") for name in top_names])

        title_text.set_text(title_template.format(time=time_value, k=k))

        if legend_ref["legend"] is not None:
            legend_ref["legend"].remove()
        handles = [
            plt.Line2D(
                [0], [0], marker="o", linestyle="",
                markerfacecolor=colors.get(name, "tab:blue"), markeredgecolor="none",
                markersize=6, label=display_sat_name(name),
            )
            for name in top_names
        ]
        legend_ref["legend"] = fig.legend(
            handles=handles,
            loc="upper right",
            bbox_to_anchor=(0.98, 0.92),
            borderaxespad=0.0,
            frameon=False,
            fontsize=7,
            handletextpad=0.3,
            labelspacing=0.25,
        )
        return scat_top, scat_other, title_text

    ani = FuncAnimation(fig, update, frames=len(time_points), interval=interval_ms, blit=False)
    out_gif = Path(out_gif)
    out_gif.parent.mkdir(parents=True, exist_ok=True)
    ani.save(out_gif, writer=PillowWriter(fps=fps), dpi=dpi)
    plt.close(fig)
    return out_gif


In [5]:
# Set this to True when you want to download fresh TLEs and regenerate visibility samples.
# Default False reuses the existing CSV, which is faster and avoids accidental network dependency.
RUN_SAMPLE = False

if RUN_SAMPLE or not cfg.visibility_csv.exists():
    ts = make_timescale()
    sats = load_starlink_tles(ts)
    visibility_df = collect_starlink_samples(sats, ts, cfg, out_csv=cfg.visibility_csv)
else:
    visibility_df = load_visibility_frame(cfg.visibility_csv)

print(f"Visibility samples: {len(visibility_df)} rows, {visibility_df['Time'].nunique()} time points")
print(f"Using visibility CSV: {cfg.visibility_csv}")
visibility_df.head()


Visibility samples: 582 rows, 30 time points
Using visibility CSV: /home/sj4025/my_project/Anti-jamming-unsync/starlink_timeseries_5min_all.csv


,Time,Name,Azimuth (°),Elevation (°),Slant km,x_East (m),y_North (m),z_Up (m)
0,2026-01-15 19:07:56,STARLINK-11655 [DTC],206.35,66.42,388.6,-69006.2,-139294.8,356207.5
1,2026-01-15 19:07:56,STARLINK-31237,331.88,66.58,525.0,-98376.1,184056.2,481723.7
2,2026-01-15 19:07:56,STARLINK-3997,326.16,77.21,554.0,-68319.7,101891.1,540205.0
3,2026-01-15 19:07:56,STARLINK-5433,4.88,75.48,557.9,11906.0,139317.1,540059.0
4,2026-01-15 19:07:56,STARLINK-33646,111.46,57.31,568.0,285505.4,-112228.0,478032.8


In [6]:
tracked_df = track_top_k(
    visibility_df,
    k=cfg.top_k,
    tol=cfg.track_tol,
    out_csv=cfg.tracked_csv,
)

print(f"Tracked samples: {len(tracked_df)} rows")
print(f"Saved tracked top-{cfg.top_k}: {cfg.tracked_csv}")
tracked_df.head(10)


Tracked samples: 150 rows
Saved tracked top-5: /home/sj4025/my_project/Anti-jamming-unsync/starlink_timeseries_5min_top5.csv


,Time,Name,Action,Rank,Slant km,Azimuth (°),Elevation (°),x_East (m),y_North (m),z_Up (m)
0,2026-01-15 19:07:56,STARLINK-11655 [DTC],seed,1,388.6,206.35,66.42,-69006.2,-139294.8,356207.5
1,2026-01-15 19:07:56,STARLINK-31237,seed,2,525.0,331.88,66.58,-98376.1,184056.2,481723.7
2,2026-01-15 19:07:56,STARLINK-3997,seed,3,554.0,326.16,77.21,-68319.7,101891.1,540205.0
3,2026-01-15 19:07:56,STARLINK-5433,seed,4,557.9,4.88,75.48,11906.0,139317.1,540059.0
4,2026-01-15 19:07:56,STARLINK-33646,seed,5,568.0,111.46,57.31,285505.4,-112228.0,478032.8
5,2026-01-15 19:08:06,STARLINK-11655 [DTC],keep,1,374.6,180.22,72.42,-430.1,-113121.6,357099.5
6,2026-01-15 19:08:06,STARLINK-11506 [DTC],add,2,474.3,289.45,47.89,-299840.4,105912.4,351840.5
7,2026-01-15 19:08:06,STARLINK-31237,keep,3,511.5,350.48,70.73,-27917.5,166491.5,482789.1
8,2026-01-15 19:08:06,STARLINK-3997,keep,4,543.9,347.16,84.06,-12509.1,54869.2,540986.5
9,2026-01-15 19:08:06,STARLINK-5433,keep,5,573.9,20.24,69.85,68392.3,185507.1,538723.6


In [7]:
static_result = save_static_sky_plot(
    visibility_df,
    out_png=cfg.static_png,
    out_pdf=cfg.static_pdf,
    k=cfg.top_k,
    elev_min_deg=cfg.elev_min_deg,
    random_seed=cfg.rng_seed,
    title=None,  # use None for IEEE-style title-free figures; use "Starlink Sky @ {time}" if needed
)

print(f"Saved static plot for {static_result['time']}: {static_result['png']}, {static_result['pdf']}")


Saved static plot for 2026-01-15 19:07:56: /home/sj4025/my_project/Anti-jamming-unsync/result_plot/starlink_sky_topk.png, /home/sj4025/my_project/Anti-jamming-unsync/result_plot/starlink_sky_topk.pdf


In [8]:
gif_path = save_sky_animation(
    visibility_df,
    out_gif=cfg.animation_gif,
    tracked_data=tracked_df,  # use the rolling tracked top-k set; pass None for nearest top-k per frame
    k=cfg.top_k,
    elev_min_deg=cfg.elev_min_deg,
    interval_ms=250,
    fps=4,
    dpi=140,
)

print(f"Saved animated top-{cfg.top_k} sky plot: {gif_path}")


Saved animated top-5 sky plot: /home/sj4025/my_project/Anti-jamming-unsync/result_plot/starlink_sky_topk.gif
